In [9]:
import json
import pandas as pd
import time
import re
import ast
import requests
from urllib.parse import urljoin, quote
from bs4 import BeautifulSoup
from urllib.parse import quote_plus
import random

In [10]:
# df view settings
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

In [11]:
DEFAULT_HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/129.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}

session = requests.Session()
session.headers.update(DEFAULT_HEADERS)


def fetch_soup(url: str) -> BeautifulSoup:
    """Fetch a page and parse it with BeautifulSoup."""
    resp = session.get(url, timeout=30)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")

In [12]:
# retrieving all the distinct car brands
u = "https://www.bilbasen.dk/brugt/bil?includeengroscvr=true&includeleasing=false"
data = json.loads(fetch_soup(u).find("script", id="__NEXT_DATA__").string)

c = []

def walk(x):
    if isinstance(x, list):
        labels = []
        for v in x:
            if isinstance(v, str):
                labels.append(v.strip())
            elif isinstance(v, dict):
                for k in ("label", "name", "title", "text", "value", "displayName"):
                    s = v.get(k)
                    if isinstance(s, str):
                        labels.append(s.strip())
                        break
        if len(labels) >= 30:
            uniq = sorted(set(labels))
            good = [
                s for s in uniq
                if s and len(s) <= 30 and s[0].isalpha() and s[0].isupper()
                and not any(ch.isdigit() for ch in s)
            ]
            if len(good) / len(uniq) > 0.8:
                c.append(uniq)
        for v in x:
            walk(v)
    elif isinstance(x, dict):
        for v in x.values():
            walk(v)

walk(data)

car_brands = sorted(c, key=len, reverse=True)[1]

In [13]:
fuel_options = {
    1: 'Benzin',
    2: 'Diesel',
    3: 'El',
    6: 'Hybrid - Benzin',
    8: 'Hybrid - Diesel',
    11: 'Plug-in Benzin',
    12: 'Plug-in Diesel'
}

In [14]:
def download_brand(brand: str, selected_fuel_type: str):
    page_listings = []

    brand_param = quote_plus(brand.strip())
    base_url = (
        f"https://www.bilbasen.dk/brugt/bil"
        f"?free={brand_param}&fuel={selected_fuel_type}&includeengroscvr=true&includeleasing=false"
    )

    soup = fetch_soup(base_url)
    page_tag = soup.find('span', {'data-e2e': 'pagination-total'})
    if not (page_tag and page_tag.text.isdigit()):
        print(f"→ Skipping {brand}: 0 pages found")
        return None

    max_page = int(page_tag.text)

    for page in range(1, max_page + 1):
        paged_url = base_url if page == 1 else f"{base_url}&page={page}"
        if page > 1:
            soup = fetch_soup(paged_url)
        print(f"Fetching {brand} page {page}/{max_page}")

        for art in soup.find_all("article"):
            if "".join(art.get("class", [])).startswith("Listing_listing"):
                for a in art.find_all("a", class_=lambda c: c and c.startswith("Listing_link")):
                    href = a.get("href")
                    if href:
                        page_listings.append(urljoin("https://www.bilbasen.dk", href))

    return page_listings


listings = []

for b in car_brands:
    pl = download_brand(b, str(selected_fuel_type))
    if pl:
        listings.extend(pl)

Fetching AC page 1/141
Fetching AC page 2/141
Fetching AC page 3/141
Fetching AC page 4/141
Fetching AC page 5/141
Fetching AC page 6/141
Fetching AC page 7/141
Fetching AC page 8/141
Fetching AC page 9/141
Fetching AC page 10/141
Fetching AC page 11/141
Fetching AC page 12/141
Fetching AC page 13/141
Fetching AC page 14/141
Fetching AC page 15/141
Fetching AC page 16/141
Fetching AC page 17/141
Fetching AC page 18/141
Fetching AC page 19/141
Fetching AC page 20/141
Fetching AC page 21/141
Fetching AC page 22/141
Fetching AC page 23/141
Fetching AC page 24/141
Fetching AC page 25/141
Fetching AC page 26/141
Fetching AC page 27/141
Fetching AC page 28/141
Fetching AC page 29/141
Fetching AC page 30/141
Fetching AC page 31/141
Fetching AC page 32/141
Fetching AC page 33/141
Fetching AC page 34/141
Fetching AC page 35/141
Fetching AC page 36/141
Fetching AC page 37/141
Fetching AC page 38/141
Fetching AC page 39/141
Fetching AC page 40/141
Fetching AC page 41/141
Fetching AC page 42/141
F

In [15]:
len(listings)


31075

In [16]:
print(len(listings), len(set(listings)))

31075 19587


In [17]:
# Getting JSON data from each listing page (avoid navigating tag hierarchies). ~ 1 minute per 100 cars
all_parsed_data = []
total = len(set(listings))
last_report = time.time()


def parse_listing_page(link: str):
    json_text = None
    soup = fetch_soup(link)
    for s in soup.find_all("script"):
        txt = (s.get_text() or "").lstrip()
        if txt.startswith("var _props"):
            m = re.search(r"var\s*_props\s*=\s*({.*?})\s*;", txt, flags=re.DOTALL)
            if m:
                json_text = m.group(1)
                break

    if not json_text:
        print("No _props JSON found on this page " + link)
        return None

    try:
        return json.loads(json_text)
    except Exception as e:
        print("Error parsing JSON:", e)
        return None


def func_wrapper_for_loop(i, link):
    global last_report

    # Progress monitoring after each 100 pages
    if i % 100 == 0 or i == total:
        now = time.time()
        elapsed = now - last_report
        mins, secs = divmod(int(elapsed), 60)
        print(
            f"{i}/{total} listings done "
            f"({i/total:.1%}) — last batch took {mins}m {secs}s",
            flush=True
        )
        last_report = now

    parsed_data = parse_listing_page(link)
    if parsed_data:
        all_parsed_data.append(parsed_data)


for i, link in enumerate(set(listings), start=1):
    func_wrapper_for_loop(i, link)
    time.sleep(0.5 + random.random())

100/19587 listings done (0.5%) — last batch took 2m 7s
200/19587 listings done (1.0%) — last batch took 1m 59s
300/19587 listings done (1.5%) — last batch took 2m 22s
400/19587 listings done (2.0%) — last batch took 1m 55s
500/19587 listings done (2.6%) — last batch took 2m 7s
600/19587 listings done (3.1%) — last batch took 2m 4s
700/19587 listings done (3.6%) — last batch took 2m 14s
800/19587 listings done (4.1%) — last batch took 2m 2s
900/19587 listings done (4.6%) — last batch took 2m 3s
1000/19587 listings done (5.1%) — last batch took 2m 3s
1100/19587 listings done (5.6%) — last batch took 2m 6s
1200/19587 listings done (6.1%) — last batch took 2m 3s
1300/19587 listings done (6.6%) — last batch took 2m 5s
1400/19587 listings done (7.1%) — last batch took 2m 3s
1500/19587 listings done (7.7%) — last batch took 1m 58s
1600/19587 listings done (8.2%) — last batch took 2m 3s
1700/19587 listings done (8.7%) — last batch took 2m 5s
1800/19587 listings done (9.2%) — last batch took 1m

In [18]:
# digest messy JSON data into a flat table of readable data
def extract_name_value(row):
    output = {}
    # Iterate over each cell in the row with its column label.
    for col, cell in row.items():
        # If the cell is a dictionary with the desired keys, transform it.
        if isinstance(cell, dict) and 'name' in cell and 'displayValue' in cell:
            output[cell['name']] = cell['displayValue']
        # If the cell is a string, try to parse it.
        elif isinstance(cell, str):
            try:
                d = ast.literal_eval(cell)
                if isinstance(d, dict) and 'name' in d and 'displayValue' in d:
                    output[d['name']] = d['displayValue']
                else:
                    # Not the desired structure, so keep the original cell under its column name.
                    output[col] = cell
            except Exception:
                # Parsing failed; keep the original cell.
                output[col] = cell
        else:
            # For any other type, simply keep the original cell.
            output[col] = cell
    return pd.Series(output)

In [19]:
all_listings = []

for entry in all_parsed_data:
    # try old key
    listing_data = entry.get("listing")

    # fall back to new path
    if listing_data is None:
        listing_data = []
        for q in (
            entry.get("props", {})
                 .get("pageProps", {})
                 .get("dehydratedState", {})
                 .get("queries", [])
        ):
            listing_data.extend(q.get("state", {}).get("data", {}).get("listings", []))

    if listing_data:
        # keep one level of nesting: 'vehicle.modelInformation' stays a dict
        flat = pd.json_normalize(listing_data, sep=".", max_level=1)
        all_listings.append(flat)

all_listings = pd.concat(all_listings, ignore_index=True)

In [20]:
# Unpacking nested dictionaries into separate columns
df_model_info = all_listings['vehicle.modelInformation'].apply(pd.Series)
df_vehicle_details = all_listings['vehicle.details'].apply(pd.Series)
df_ratings = all_listings['vehicle.ratings'].apply(pd.Series)
df_seller_address = all_listings['seller.address'].apply(pd.Series)
df_equipment = all_listings['vehicle.equipment'].apply(pd.Series)

df_otheritems = all_listings['seller.sellerOtherItems'].apply(pd.Series)

df_values_only = df_equipment.applymap(lambda x: x.get('value') if isinstance(x, dict) else None)
df_binarized = (
    df_values_only
    .stack()
    .str.get_dummies()
    .groupby(level=0)
    .max()
)
df_binarized = df_binarized.fillna(0).astype(int)

df_base = all_listings.drop(['vehicle.modelInformation', 'vehicle.details', 'vehicle.ratings', 'seller.address', 'vehicle.equipment', 'seller.sellerOtherItems'], axis=1)
df_expanded = pd.concat([df_base, df_model_info, df_vehicle_details, df_ratings, df_seller_address, df_otheritems, df_binarized], axis=1)

/tmp/ipykernel_2491/1848402279.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_values_only = df_equipment.applymap(lambda x: x.get('value') if isinstance(x, dict) else None)


In [35]:
rows = [extract_name_value(row) for _, row in df_expanded.iterrows()]
df_result = pd.DataFrame(rows)

<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal


In [36]:
df_result

,externalId,syiId,tenant,canonicalUrl,description,alternativeListings,price.name,price.displayValue,vehicle.make,vehicle.model,vehicle.variant,vehicle.modelYear,media.images,media.fullscreenImages,seller.id,seller.name,seller.type,seller.tenant,seller.phoneNumbers,seller.features,seller.hasEmail,seller.phoneNumberType,seller.dsaText,media.video,features.carOfTheYear,price.description,Nypris,Kategori,Type,Bagagerumsstørrelse,Vægt,Bredde,Længde,Højde,Lasteevne,Max. trækvægt m/bremse,Trækhjul,ABS-bremser,ESP,Airbags,Døre,Modelår,1. registrering,Kilometertal,Drivmiddel,Rækkevidde,Batterikapacitet,Energiforbrug,Hjemmeopladning AC,Hurtig opladning DC,Opladningstid DC 10-80%,Periodisk afgift,Ydelse,Acceleration,Tophastighed,Trækvægt,Farve,16,average,numberOfReviews,subRatings,label,0,line1,line2,zipCode,city,country,url,numberOfListings,1 ejer,10 airbags,15 tommer Alufælge,16 tommer Alufælge,17 tommer Alufælge,18 tommer Alufælge,19 tommer Alufælge,2 zone klima,20 tommer Alufælge,21 tommer Alufælge,22 tommer Alufælge,3 individuelle sæder i bag,3 zone klima,360° kamera,4 airbags,4 zone klima,4x el-ruder,6 airbags,6 gear,6 personers,7 airbags,7 personers,8 airbags,9 airbags,AUX tilslutning,Adaptiv fartpilot,Adaptiv fartpilot med kø-assistent,Adaptiv undervogn,Adaptive forlygter,Aftageligt træk,Aircondition,Akustikglas i bag,Akustikglas i for,Alarm,Alcantaraindtræk,Alufælge,Ambiente belysning,Android auto,"Anhængertræk, aftagl.",Antispin,App integration,Apple carplay,Armlæn,Aut. nedbl. bakspejl,Aut.gear/tiptronic,Auto hold,Auto. nødbremse,Auto. parkering,Automatgear,Automatisk forvarmning af batteri,Automatisk lys,Automatisk nødassistent,Automatisk nødbremsesystem,Automatisk parkeringssystem,Automatisk start/stop,Bagagerumsdækken,Bagsæde underholdningssystem,Bakkamera,Bakspejl m. nedbl.,Bi-xenon,Blindvinkelassistent,Bluetooth,Brugtbilsattest,CD,CD afspiller,CD/radio,Centrallås,City steering,DAB radio,DAB+ Radio,Delalcantaraindtræk,Delkunstlæderindtræk,Dellæderindtræk,Diesel partikel filter,Digitalt cockpit,Dobbelt bagagerumsbund,Dynamisk blinklys,Dynamisk blinklys i bag,Dæktryksmåler,EDR-boks,ESP / ESC,El betjent bagklap,El betjent frontklap,El betjente døre,El indst. forsæder,El indst. førersæde,El indst. førersæde m. memory,El komfortsæder,El-justerbar lændestøtte,El-klapbare sidespejle,El-klapbare sidespejle m. varme,El-ruder,El-sidespejle,El-sidespejle m. varme,El-soltag,Elektrisk kabinevarmer,Elektrisk parkeringsbremse,Elsæder,Fartpilot,Fjernb. centrallås,Fjernlysassistent,Fod betjent bagklap,Frunk,Fuld LED forlygter,Fuldaut. klima,Førerovervågning med advarsel,Head-up display,Helårshjul,Hvide blink,Håndfrit til mobil,Hækspoiler,Højdejust. forsæde,Højdejust. førersæde,Ikke ryger,Indfarvede kofangere,Infocenter,Integrerede børnesæder,Integrerede rullegardiner,Integreret ladekabel,Intelligent hastighedsassistent,"Interiør, træ",Internet,Isofix,Justerbar lændestøtte,"Klimaanlæg, 2-zonet","Klimaanlæg, 3-zonet","Klimaanlæg, 4-zonet",Kopholder,Kunstlæder,Kunstlæderindtræk,Kurvelys,Køl i handskerum,Kørecomputer,LED Kørelys,LED baglygter,LED forlygter,Laser forlygter,Lev. nysynet,Luftundervogn,Lygtevasker,Læderindtræk,Læderrat,Manuel forvarmning af batteri,Massage i forsæder,Massage i førersæde,Matrix LED forlygter,Motorkabinevarmer,Multifunktionsrat,Musikstreaming via bluetooth,Mørk loftbeklædning,Mørktonede ruder i bag,Navigation,Nightvision,Nysynet,Nødlader,Nøglefri adgang,Nøglefri betjening,Nøglefri tænding,Områdebelysning,Panoramatag,Parkeringssensor,Parkeringssensor (bag),Parkeringssensor (for),Pilotsæder,Radio med CD-boks,Ratbetjent gearskifte,Ratgearskifte,Regnsensor,SD kortlæser,Semi-automatisk parkeringssystem,Service ok,Servo,Skiltegenkendelse,Skørter,Solceller,Soltag,Splitbagsæde,Spoiler,Sportssæder,Startspærre,Stemmebetjening,Stofindtræk,Svingbart træk (elektrisk),Svingbart træk (manuelt),"Sædebetræk, dellæder","Sædebetræk, læder","Sædebetræk, stof",Sædekøling,Sædevarme,Sænket,Tagræling,Tidligere undervognsbehandlet,Tonede rude

In [37]:
accessories_cols = df_binarized.columns.tolist()

In [38]:
benzin_cols = [
        'scrape_timestamp','price.displayValue', 'Nypris', 'vehicle.make', 'vehicle.model', 'vehicle.variant', 'vehicle.modelYear', '1. registrering', 
        'Kilometertal', 'Ydelse', 'Acceleration', 'Tophastighed', 'Trækvægt', 'Farve',
        'Kategori', 'Type', 'Bagagerumsstørrelse', 'Vægt', 'Bredde', 'Længde', 'Højde', 'Lasteevne', 'Max. trækvægt m/bremse', 'Trækhjul',
        'Drivmiddel', 'Brændstofforbrug','Cylindre', 'Airbags', 'Tankkapacitet','ABS-bremser', 'ESP', 'Periodisk afgift','CO2 udledning', 'Euronorm', 
        'price.description', 'seller.name', 'zipCode', 'city', 'numberOfListings',
        'average', 'numberOfReviews', 'canonicalUrl','externalId']
benzin_cols += accessories_cols

el_cols = [
        'scrape_timestamp','price.displayValue', 'Nypris', 'vehicle.make', 'vehicle.model', 'vehicle.variant', 'vehicle.modelYear', '1. registrering', 
        'Kilometertal', 'Ydelse', 'Acceleration', 'Tophastighed', 'Trækvægt', 'Farve',
        'Kategori', 'Type', 'Bagagerumsstørrelse', 'Vægt', 'Bredde', 'Længde', 'Højde', 'Lasteevne', 'Max. trækvægt m/bremse', 'Trækhjul',
        'Drivmiddel', 'Energiforbrug', 'Batterikapacitet', 'Rækkevidde', 'Hjemmeopladning AC', 'Hurtig opladning DC', 'Opladningstid DC 10-80%',
        'Airbags', 'ABS-bremser', 'ESP', 'Døre', 'Periodisk afgift', 
        'price.description', 'seller.name', 'price.description', 'seller.name', 'zipCode', 'city', 'numberOfListings',
        'average', 'numberOfReviews', 'canonicalUrl','externalId']
el_cols += accessories_cols

In [ ]:
benzin_cols = [
        'scrape_timestamp','price.displayValue', 'Nypris', 'vehicle.make', 'vehicle.model', 'vehicle.variant', 'vehicle.modelYear', '1. registrering', 
        'Kilometertal', 'Ydelse', 'Acceleration', 'Tophastighed', 'Trækvægt', 'Farve',
        'Kategori', 'Type', 'Bagagerumsstørrelse', 'Vægt', 'Bredde', 'Længde', 'Højde', 'Lasteevne', 'Max. trækvægt m/bremse', 'Trækhjul',
        'Drivmiddel', 'Brændstofforbrug','Cylindre', 'Airbags', 'Tankkapacitet','ABS-bremser', 'ESP', 'Periodisk afgift','CO2 udledning', 'Euronorm', 
        'price.description', 'seller.name', 'zipCode', 'city', 'numberOfListings',
        'average', 'numberOfReviews', 'canonicalUrl','externalId',

        '1 ejer','10 airbags','15 tommer Alufælge','16 tommer Alufælge','17 tommer Alufælge','18 tommer Alufælge','19 tommer Alufælge','2 zone klima',
        '20 tommer Alufælge','21 tommer Alufælge','22 tommer Alufælge','3 individuelle sæder i bag','3 zone klima','360° kamera','4 airbags','4 zone klima',
        '4x el-ruder','6 airbags','6 gear','6 personers','7 airbags','7 personers','8 airbags','9 airbags','ABS-bremser','AUX tilslutning','Adaptiv fartpilot',
        'Adaptiv fartpilot med kø-assistent','Adaptiv undervogn','Adaptive forlygter','Afhentning','Aftageligt træk','Airbags','Aircondition','Akustikglas i bag',
        'Akustikglas i for','Alarm','Alcantaraindtræk','Alufælge','Ambiente belysning','Android auto','Anhængertræk','Anhængertræk, aftagl.','Antispin','App integration',
        'Apple carplay','Armlæn','Aut. nedbl. bakspejl','Aut.gear/tiptronic','Auto hold','Auto. nødbremse','Auto. parkering','Auto. start/stop','Automatgear',
        'Automatisk forvarmning af batteri','Automatisk lys','Automatisk nødassistent','Automatisk nødbremsesystem','Automatisk parkeringssystem','Automatisk start/stop',
        'Bagagerumsdækken','Bagsæde underholdningssystem','Bakkamera','Bakspejl m. nedbl.','Bi-xenon','Blindvinkelassistent','Bluetooth','Brugtbilsattest','CD','CD afspiller',
        'CD/radio','Centrallås','Centrallås fjernb.','City steering','DAB radio','DAB+ Radio','Delalcantaraindtræk','Delkunstlæderindtræk','Dellæderindtræk',
        'Diesel partikel filter','Digitalt cockpit','Dobbelt bagagerumsbund','Dynamisk blinklys','Dynamisk blinklys i bag','Dæktryksmåler','EDR-boks','ESP / ESC',
        'El betjent bagklap','El betjent frontklap','El betjente døre','El indst. forsæder','El indst. førersæde','El indst. førersæde m. memory','El komfortsæder',
        'El-justerbar lændestøtte','El-klapbare sidespejle','El-klapbare sidespejle m. varme','El-ruder','El-sidespejle','El-sidespejle m. varme','El-soltag',
        'Elektrisk kabinevarmer','Elektrisk parkeringsbremse','Elektronisk bagklap','Elruder, 4x','Elsidespejle','Elsidespejle m. varme','Elsæder','Fartpilot',
        'Fartpilot, adaptiv','Fjernb. centrallås','Fjernlysassistent','Fod betjent bagklap','Frunk','Fuld LED forlygter','Fuldaut. klima','Førerovervågning med advarsel',
        'Head-up display','Helårshjul','Hvide blink','Håndfrit til mobil','Hækspoiler','Højdejust. forsæde','Højdejust. førersæde','Ikke ryger','Indfarvede kofangere',
        'Infocenter','Integrerede børnesæder','Integrerede rullegardiner','Intelligent hastighedsassistent','Internet','Isofix','Justerbar lændestøtte','Klimaanlæg','Klimaanlæg, 2-zonet',
        'Klimaanlæg, 3-zonet','Klimaanlæg, 4-zonet','Kopholder','Kunstlæder','Kunstlæderindtræk','Kurvelys','Køl i handskerum','Kørecomputer','LED Kørelys','LED baglygter','LED forlygter',
        'Laser forlygter','Lev. nysynet','Luftundervogn','Lygtevasker','Læderindtræk','Læderrat','Manuel forvarmning af batteri','Massage i forsæder','Massage i førersæde',
        'Matrix LED forlygter','Motorkabinevarmer','Multifunktionsrat','Musikstreaming via bluetooth','Mørk loftbeklædning','Mørktonede ruder i bag','Navigation','Nightvision',
        'Nysynet','Nødlader','Nøglefri adgang','Nøglefri betjening','Nøglefri tænding','Områdebelysning','Panoramatag','Parkeringssensor','Parkeringssensor (bag)','Parkeringssensor (for)',
        'Pilotsæder','Radio med CD-boks','Ratbetjent gearskifte','Ratgearskifte','Regnsensor','SD kortlæser','Semi-automatisk parkeringssystem','Service ok','Servo','Skiltegenkendelse',
        'Skørter','Solceller','Soltag','Splitbagsæde','Spoiler','Sportssæder','Startspærre','Stemmebetjening','Stofindtræk','Svingbart træk (elektrisk)','Svingbart træk (manuelt)',
        'Sædebetræk, dellæder','Sædebetræk, læder','Sædebetræk, stof','Sædekøling','Sædevarme','Sænket','Tagræling','Tidligere undervognsbehandlet','Tonede ruder','Trafikkamera',
        'Trådløs mobilopladning','Træk','Træthedsregistrering','Type-2 ladekabel','Tågelygter','USB A tilslutning','USB C tilslutning','USB tilslutning','Udv. temp. måler','Undervogn, sænket',
        'Varme i 3. sæderække','Varme i bagsæde','Varme i forrude','Varme i rat','Varmepumpe','Videoovervågning','Vinterhjul','Virtuelle sidespejle','Virtuelt bakspejl','Vognbaneassistent','Xenonlygter'
]

el_cols = [
        'scrape_timestamp','price.displayValue', 'Nypris', 'vehicle.make', 'vehicle.model', 'vehicle.variant', 'vehicle.modelYear', '1. registrering', 
        'Kilometertal', 'Ydelse', 'Acceleration', 'Tophastighed', 'Trækvægt', 'Farve',
        'Kategori', 'Type', 'Bagagerumsstørrelse', 'Vægt', 'Bredde', 'Længde', 'Højde', 'Lasteevne', 'Max. trækvægt m/bremse', 'Trækhjul',
        'Drivmiddel', 'Energiforbrug', 'Batterikapacitet', 'Rækkevidde', 'Hjemmeopladning AC', 'Hurtig opladning DC', 'Opladningstid DC 10-80%',
        'Airbags', 'ABS-bremser', 'ESP', 'Døre', 'Periodisk afgift', 
        'price.description', 'seller.name', 'price.description', 'seller.name', 'zipCode', 'city', 'numberOfListings',
        'average', 'numberOfReviews', 'canonicalUrl','externalId',

        '1 ejer','10 airbags','15 tommer Alufælge','16 tommer Alufælge','17 tommer Alufælge','18 tommer Alufælge','19 tommer Alufælge','2 zone klima',
        '20 tommer Alufælge','21 tommer Alufælge','22 tommer Alufælge','3 individuelle sæder i bag','3 zone klima','360° kamera','4 airbags','4 zone klima',
        '4x el-ruder','6 airbags','6 gear','6 personers','7 airbags','7 personers','8 airbags','9 airbags','ABS-bremser','AUX tilslutning','Adaptiv fartpilot',
        'Adaptiv fartpilot med kø-assistent','Adaptiv undervogn','Adaptive forlygter','Afhentning','Aftageligt træk','Airbags','Aircondition','Akustikglas i bag',
        'Akustikglas i for','Alarm','Alcantaraindtræk','Alufælge','Ambiente belysning','Android auto','Anhængertræk','Anhængertræk, aftagl.','Antispin','App integration',
        'Apple carplay','Armlæn','Aut. nedbl. bakspejl','Aut.gear/tiptronic','Auto hold','Auto. nødbremse','Auto. parkering','Auto. start/stop','Automatgear',
        'Automatisk forvarmning af batteri','Automatisk lys','Automatisk nødassistent','Automatisk nødbremsesystem','Automatisk parkeringssystem','Automatisk start/stop',
        'Bagagerumsdækken','Bagsæde underholdningssystem','Bakkamera','Bakspejl m. nedbl.','Bi-xenon','Blindvinkelassistent','Bluetooth','Brugtbilsattest','CD','CD afspiller',
        'CD/radio','Centrallås','Centrallås fjernb.','City steering','DAB radio','DAB+ Radio','Delalcantaraindtræk','Delkunstlæderindtræk','Dellæderindtræk',
        'Diesel partikel filter','Digitalt cockpit','Dobbelt bagagerumsbund','Dynamisk blinklys','Dynamisk blinklys i bag','Dæktryksmåler','EDR-boks','ESP / ESC',
        'El betjent bagklap','El betjent frontklap','El betjente døre','El indst. forsæder','El indst. førersæde','El indst. førersæde m. memory','El komfortsæder',
        'El-justerbar lændestøtte','El-klapbare sidespejle','El-klapbare sidespejle m. varme','El-ruder','El-sidespejle','El-sidespejle m. varme','El-soltag',
        'Elektrisk kabinevarmer','Elektrisk parkeringsbremse','Elektronisk bagklap','Elruder, 4x','Elsidespejle','Elsidespejle m. varme','Elsæder','Fartpilot',
        'Fartpilot, adaptiv','Fjernb. centrallås','Fjernlysassistent','Fod betjent bagklap','Frunk','Fuld LED forlygter','Fuldaut. klima','Førerovervågning med advarsel',
        'Head-up display','Helårshjul','Hvide blink','Håndfrit til mobil','Hækspoiler','Højdejust. forsæde','Højdejust. førersæde','Ikke ryger','Indfarvede kofangere',
        'Infocenter','Integrerede børnesæder','Integrerede rullegardiner','Intelligent hastighedsassistent','Internet','Isofix','Justerbar lændestøtte','Klimaanlæg','Klimaanlæg, 2-zonet',
        'Klimaanlæg, 3-zonet','Klimaanlæg, 4-zonet','Kopholder','Kunstlæder','Kunstlæderindtræk','Kurvelys','Køl i handskerum','Kørecomputer','LED Kørelys','LED baglygter','LED forlygter',
        'Laser forlygter','Lev. nysynet','Luftundervogn','Lygtevasker','Læderindtræk','Læderrat','Manuel forvarmning af batteri','Massage i forsæder','Massage i førersæde',
        'Matrix LED forlygter','Motorkabinevarmer','Multifunktionsrat','Musikstreaming via bluetooth','Mørk loftbeklædning','Mørktonede ruder i bag','Navigation','Nightvision',
        'Nysynet','Nødlader','Nøglefri adgang','Nøglefri betjening','Nøglefri tænding','Områdebelysning','Panoramatag','Parkeringssensor','Parkeringssensor (bag)','Parkeringssensor (for)',
        'Pilotsæder','Radio med CD-boks','Ratbetjent gearskifte','Ratgearskifte','Regnsensor','SD kortlæser','Semi-automatisk parkeringssystem','Service ok','Servo','Skiltegenkendelse',
        'Skørter','Solceller','Soltag','Splitbagsæde','Spoiler','Sportssæder','Startspærre','Stemmebetjening','Stofindtræk','Svingbart træk (elektrisk)','Svingbart træk (manuelt)',
        'Sædebetræk, dellæder','Sædebetræk, læder','Sædebetræk, stof','Sædekøling','Sædevarme','Sænket','Tagræling','Tidligere undervognsbehandlet','Tonede ruder','Trafikkamera',
        'Trådløs mobilopladning','Træk','Træthedsregistrering','Type-2 ladekabel','Tågelygter','USB A tilslutning','USB C tilslutning','USB tilslutning','Udv. temp. måler','Undervogn, sænket',
        'Varme i 3. sæderække','Varme i bagsæde','Varme i forrude','Varme i rat','Varmepumpe','Videoovervågning','Vinterhjul','Virtuelle sidespejle','Virtuelt bakspejl','Vognbaneassistent','Xenonlygter'
    ]

In [39]:
columns = {
    'Benzin': benzin_cols,
    'Diesel': benzin_cols,
    'El':     el_cols,

}

In [40]:
today = pd.Timestamp.now().replace(microsecond=0)
yesterday = today - pd.Timedelta(days=1)
print(today, yesterday)
df_result.insert(0, 'scrape_timestamp', today)

2025-12-19 06:56:48 2025-12-18 06:56:48


In [41]:
df = df_result[columns[fuel_options[selected_fuel_type]]].copy()

In [42]:
df = df.loc[:, ~df.columns.duplicated()]

In [43]:
df

,scrape_timestamp,price.displayValue,Nypris,vehicle.make,vehicle.model,vehicle.variant,vehicle.modelYear,1. registrering,Kilometertal,Ydelse,Acceleration,Tophastighed,Trækvægt,Farve,Kategori,Type,Bagagerumsstørrelse,Vægt,Bredde,Længde,Højde,Lasteevne,Max. trækvægt m/bremse,Trækhjul,Drivmiddel,Energiforbrug,Batterikapacitet,Rækkevidde,Hjemmeopladning AC,Hurtig opladning DC,Opladningstid DC 10-80%,Airbags,ABS-bremser,ESP,Døre,Periodisk afgift,price.description,seller.name,zipCode,city,numberOfListings,average,numberOfReviews,canonicalUrl,externalId,1 ejer,10 airbags,15 tommer Alufælge,16 tommer Alufælge,17 tommer Alufælge,18 tommer Alufælge,19 tommer Alufælge,2 zone klima,20 tommer Alufælge,21 tommer Alufælge,22 tommer Alufælge,3 individuelle sæder i bag,3 zone klima,360° kamera,4 airbags,4 zone klima,4x el-ruder,6 airbags,6 gear,6 personers,7 airbags,7 personers,8 airbags,9 airbags,AUX tilslutning,Adaptiv fartpilot,Adaptiv fartpilot med kø-assistent,Adaptiv undervogn,Adaptive forlygter,Aftageligt træk,Aircondition,Akustikglas i bag,Akustikglas i for,Alarm,Alcantaraindtræk,Alufælge,Ambiente belysning,Android auto,"Anhængertræk, aftagl.",Antispin,App integration,Apple carplay,Armlæn,Aut. nedbl. bakspejl,Aut.gear/tiptronic,Auto hold,Auto. nødbremse,Auto. parkering,Automatgear,Automatisk forvarmning af batteri,Automatisk lys,Automatisk nødassistent,Automatisk nødbremsesystem,Automatisk parkeringssystem,Automatisk start/stop,Bagagerumsdækken,Bagsæde underholdningssystem,Bakkamera,Bakspejl m. nedbl.,Bi-xenon,Blindvinkelassistent,Bluetooth,Brugtbilsattest,CD,CD afspiller,CD/radio,Centrallås,City steering,DAB radio,DAB+ Radio,Delalcantaraindtræk,Delkunstlæderindtræk,Dellæderindtræk,Diesel partikel filter,Digitalt cockpit,Dobbelt bagagerumsbund,Dynamisk blinklys,Dynamisk blinklys i bag,Dæktryksmåler,EDR-boks,ESP / ESC,El betjent bagklap,El betjent frontklap,El betjente døre,El indst. forsæder,El indst. førersæde,El indst. førersæde m. memory,El komfortsæder,El-justerbar lændestøtte,El-klapbare sidespejle,El-klapbare sidespejle m. varme,El-ruder,El-sidespejle,El-sidespejle m. varme,El-soltag,Elektrisk kabinevarmer,Elektrisk parkeringsbremse,Elsæder,Fartpilot,Fjernb. centrallås,Fjernlysassistent,Fod betjent bagklap,Frunk,Fuld LED forlygter,Fuldaut. klima,Førerovervågning med advarsel,Head-up display,Helårshjul,Hvide blink,Håndfrit til mobil,Hækspoiler,Højdejust. forsæde,Højdejust. førersæde,Ikke ryger,Indfarvede kofangere,Infocenter,Integrerede børnesæder,Integrerede rullegardiner,Integreret ladekabel,Intelligent hastighedsassistent,"Interiør, træ",Internet,Isofix,Justerbar lændestøtte,"Klimaanlæg, 2-zonet","Klimaanlæg, 3-zonet","Klimaanlæg, 4-zonet",Kopholder,Kunstlæder,Kunstlæderindtræk,Kurvelys,Køl i handskerum,Kørecomputer,LED Kørelys,LED baglygter,LED forlygter,Laser forlygter,Lev. nysynet,Luftundervogn,Lygtevasker,Læderindtræk,Læderrat,Manuel forvarmning af batteri,Massage i forsæder,Massage i førersæde,Matrix LED forlygter,Motorkabinevarmer,Multifunktionsrat,Musikstreaming via bluetooth,Mørk loftbeklædning,Mørktonede ruder i bag,Navigation,Nightvision,Nysynet,Nødlader,Nøglefri adgang,Nøglefri betjening,Nøglefri tænding,Områdebelysning,Panoramatag,Parkeringssensor,Parkeringssensor (bag),Parkeringssensor (for),Pilotsæder,Radio med CD-boks,Ratbetjent gearskifte,Ratgearskifte,Regnsensor,SD kortlæser,Semi-automatisk parkeringssystem,Service ok,Servo,Skiltegenkendelse,Skørter,Solceller,Soltag,Splitbagsæde,Spoiler,Sportssæder,Startspærre,Stemmebetjening,Stofindtræk,Svingbart træk (elektrisk),Svingbart træk (manuelt),"Sædebetræk, dellæder","Sædebetræk, læder","Sædebetræk, stof",Sædekøling,Sædevarme,Sænket,Tagræling,Tidligere undervognsbehandlet,Tonede ruder,Trafikkamera,Trådløs mobilopladning,Træk,Træthedsregistrering,Type-2 ladekabel,Tågelygter,USB A tilslutning,USB C tilslutning,USB tilslutning,Udv. temp. måler,"Undervogn, sænket",V2G,V2L,Varme i 3. sæderække,Varme i bagsæde,Varme i forrude,Varme i rat,Varmepumpe,Videoovervågning,Vinterh

In [44]:
today_str = today.strftime("%Y-%m-%d")

df.to_parquet(
    f"/home/pi-vault/projects/bilbasen_webscraping/data/{fuel_options[selected_fuel_type]}/"
    f"{fuel_options[selected_fuel_type]}_listings_{today_str}.parquet",
    index=True,
    engine="pyarrow",
)